# Stage 2 — Defect Segmentation Baseline

**任務**：semantic segmentation，3 類 (背景 / 正常零件 / 瑕疵零件)，per-pixel 分類。

**資料**：1000 張 256×256 RGB 場景（多 pan_head 螺絲在工業背景上，允許重疊），ground truth 為 pixel-level mask。

**模型**：自製 Encoder-Decoder + skip connections（4 stage encoder/decoder，~5M params）。**不用 pretrained，不抄 U-Net。**

**指標**：per-class IoU、mean IoU、pixel accuracy；後處理算 per-instance precision/recall via connected components。

In [ ]:
# ─── Setup ───
import os, json, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}', '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ─── Dataset ───
SCENES_DIR = r'D:\Harrison\中山\大四下\深度學習期末報告\output\scenes'
NUM_CLASSES = 3   # 0=bg, 1=normal_part, 2=defective_part
CLASS_NAMES = ['background', 'normal_part', 'defective_part']

# ─── QUICK_TEST：True 用 30 張、2 epoch 估算單 epoch 時間；False 跑完整 ───
QUICK_TEST = True

class SceneDataset(Dataset):
    def __init__(self, root, scene_ids):
        self.root = root
        self.scene_ids = scene_ids
    def __len__(self):
        return len(self.scene_ids)
    def __getitem__(self, idx):
        sid = self.scene_ids[idx]
        scene_dir = os.path.join(self.root, f'{sid:05d}')
        rgb = np.array(Image.open(os.path.join(scene_dir, 'rgb.png')).convert('RGB'))
        mask = np.array(Image.open(os.path.join(scene_dir, 'semantic_mask.png')))
        rgb_t = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
        rgb_t = (rgb_t - 0.5) / 0.5
        mask_t = torch.from_numpy(mask).long()
        return rgb_t, mask_t

all_ids = sorted(int(os.path.basename(d)) for d in glob.glob(os.path.join(SCENES_DIR, '[0-9]*')))
print(f'Total scenes available: {len(all_ids)}')

rng = np.random.RandomState(SEED)
shuffled = list(all_ids); rng.shuffle(shuffled)
if QUICK_TEST:
    shuffled = shuffled[:30]
    print('*** QUICK_TEST mode：只用 30 張 ***')
n_train = int(0.8 * len(shuffled))
n_val   = int(0.1 * len(shuffled))
train_ids = shuffled[:n_train]
val_ids   = shuffled[n_train:n_train + n_val]
test_ids  = shuffled[n_train + n_val:]
print(f'Train / Val / Test: {len(train_ids)} / {len(val_ids)} / {len(test_ids)}')

BATCH_SIZE = 8
train_loader = DataLoader(SceneDataset(SCENES_DIR, train_ids), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(SceneDataset(SCENES_DIR, val_ids),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(SceneDataset(SCENES_DIR, test_ids),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:
# ─── 視覺檢查 + 計算 class weight ───
def unnorm(t):
    return ((t * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0).numpy())

def mask_to_color(m):
    # 0=bg(黑), 1=normal(綠), 2=defect(紅)
    cmap = np.array([[0,0,0], [0,200,0], [200,0,0]], dtype=np.uint8)
    return cmap[m]

rgb, mask = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(unnorm(rgb[i])); axes[0, i].set_title(f'Scene {i}'); axes[0, i].axis('off')
    axes[1, i].imshow(mask_to_color(mask[i].numpy())); axes[1, i].set_title('GT mask'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

# 估 class weight（用 100 個 sample）
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for i in range(min(100, len(train_ids))):
    _, m = SceneDataset(SCENES_DIR, train_ids)[i]
    for c in range(NUM_CLASSES):
        counts[c] += int((m == c).sum())
freq = counts / counts.sum()
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * NUM_CLASSES
print(f'Class frequencies: {freq}')
print(f'Class weights:     {weights}')
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)

In [ ]:
# ─── 自製 Encoder-Decoder 架構 ───
# 設計原則：
#   * 4 stage 對稱 encoder-decoder
#   * 每 stage 兩層 Conv 3x3 + BatchNorm + ReLU
#   * skip connections：encoder 同 stage 的 feature concat 到 decoder
#   * Upsample 用 transposed conv（自製、不用 nearest neighbor）
#   * 最後 1x1 conv 投影到 num_classes
# 不抄 U-Net 任何具體 channel 設定；我們自己決定 32→64→128→256，深度 4 stage

class ConvBlock(nn.Module):
    """兩層 3x3 Conv + BN + ReLU"""
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = F.relu(self.bn2(self.conv2(x)), inplace=True)
        return x

class DefectSegNet(nn.Module):
    def __init__(self, num_classes=3, base_c=32):
        super().__init__()
        c = base_c
        # Encoder (4 stages with maxpool between)
        self.enc1 = ConvBlock(3,      c)        # 256×256, c
        self.enc2 = ConvBlock(c,      c*2)      # 128×128, 2c
        self.enc3 = ConvBlock(c*2,    c*4)      # 64×64, 4c
        self.enc4 = ConvBlock(c*4,    c*8)      # 32×32, 8c
        self.pool = nn.MaxPool2d(2)
        # Bottleneck
        self.bottleneck = ConvBlock(c*8, c*8)   # 16×16, 8c
        # Decoder (4 stages, upsample via transposed conv, concat skip from encoder)
        self.up4  = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dec4 = ConvBlock(c*4 + c*8, c*4)
        self.up3  = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec3 = ConvBlock(c*2 + c*4, c*2)
        self.up2  = nn.ConvTranspose2d(c*2, c,   2, stride=2)
        self.dec2 = ConvBlock(c + c*2,   c)
        self.up1  = nn.ConvTranspose2d(c,   c//2, 2, stride=2)
        self.dec1 = ConvBlock(c//2 + c,  c//2)
        self.out_conv = nn.Conv2d(c//2, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)        # (B, c,   256, 256)
        e2 = self.enc2(self.pool(e1))  # (B, 2c,  128, 128)
        e3 = self.enc3(self.pool(e2))  # (B, 4c,  64,  64)
        e4 = self.enc4(self.pool(e3))  # (B, 8c,  32,  32)
        b  = self.bottleneck(self.pool(e4))  # (B, 8c, 16, 16)
        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))   # (B, 4c, 32,  32)
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))   # (B, 2c, 64,  64)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))   # (B, c,  128, 128)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))   # (B, c/2,256, 256)
        return self.out_conv(d1)  # (B, num_classes, 256, 256)

model = DefectSegNet(num_classes=NUM_CLASSES, base_c=32).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

In [ ]:
# ─── Train / Val helpers ───
def compute_iou(pred, target, num_classes):
    """Per-class IoU。pred, target: (B,H,W) long tensor on device"""
    ious = []
    for c in range(num_classes):
        p = (pred == c)
        t = (target == c)
        inter = (p & t).sum().item()
        union = (p | t).sum().item()
        ious.append(inter / union if union > 0 else float('nan'))
    return ious

def evaluate(model, loader, num_classes=NUM_CLASSES):
    model.eval()
    inter = np.zeros(num_classes); union = np.zeros(num_classes)
    correct = total = 0
    with torch.no_grad():
        for rgb, mask in loader:
            rgb, mask = rgb.to(device), mask.to(device)
            pred = model(rgb).argmax(dim=1)
            for c in range(num_classes):
                p = (pred == c); t = (mask == c)
                inter[c] += (p & t).sum().item()
                union[c] += (p | t).sum().item()
            correct += (pred == mask).sum().item()
            total   += mask.numel()
    ious  = [inter[c]/union[c] if union[c]>0 else float('nan') for c in range(num_classes)]
    miou  = np.nanmean(ious)
    pacc  = correct / total
    return {'pixel_acc': pacc, 'mIoU': miou, 'IoU_per_class': ious}

def train_one_epoch(model, optimizer, loader, criterion):
    model.train()
    running_loss = 0.0
    for rgb, mask in loader:
        rgb, mask = rgb.to(device), mask.to(device)
        logits = model(rgb)  # (B, C, H, W)
        loss = criterion(logits, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# ─── 訓練 ───
import time
EPOCHS = 2 if QUICK_TEST else 30
LR = 0.01

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

history = {'train_loss': [], 'val_miou': [], 'val_iou_bg': [], 'val_iou_normal': [], 'val_iou_defect': []}
best_miou = 0.0
best_state = None
epoch_times = []

for ep in range(EPOCHS):
    t0 = time.time()
    loss = train_one_epoch(model, optimizer, train_loader, criterion)
    train_dt = time.time() - t0
    t1 = time.time()
    val_metrics = evaluate(model, val_loader)
    val_dt = time.time() - t1
    epoch_times.append(train_dt)
    history['train_loss'].append(loss)
    history['val_miou'].append(val_metrics['mIoU'])
    history['val_iou_bg'].append(val_metrics['IoU_per_class'][0])
    history['val_iou_normal'].append(val_metrics['IoU_per_class'][1])
    history['val_iou_defect'].append(val_metrics['IoU_per_class'][2])
    if val_metrics['mIoU'] > best_miou:
        best_miou = val_metrics['mIoU']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    ious = val_metrics['IoU_per_class']
    print(f'Epoch {ep+1:2d}/{EPOCHS} [train={train_dt:.1f}s val={val_dt:.1f}s] loss={loss:.4f} mIoU={val_metrics["mIoU"]:.3f} bg={ious[0]:.3f} norm={ious[1]:.3f} def={ious[2]:.3f}')

if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\nLoaded best model: mIoU={best_miou:.3f}')

# 推算完整訓練時間
if QUICK_TEST:
    avg = sum(epoch_times) / len(epoch_times)
    n_train_full = int(0.8 * 1000)
    scale = n_train_full / len(train_ids)
    est_full_min = avg * scale * 30 / 60
    print(f'\n[推算] 完整 (800 train × 30 ep) ~= {est_full_min:.1f} 分鐘')
    print(f'      若改 base_c=16 模型 (~1/4 params)，再快約 2-3 倍 ~= {est_full_min/2.5:.1f} 分鐘')


In [ ]:
# ─── 學習曲線 ───
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], 'b-')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss'); axes[0].grid(True)
axes[0].set_title('Training Loss')

axes[1].plot(history['val_miou'],       label='mIoU',          linewidth=2)
axes[1].plot(history['val_iou_bg'],     label='IoU bg',        linestyle='--')
axes[1].plot(history['val_iou_normal'], label='IoU normal',    linestyle='--')
axes[1].plot(history['val_iou_defect'], label='IoU defect',    linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU'); axes[1].grid(True); axes[1].legend()
axes[1].set_title('Validation IoU')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Test set 最終評估 ───
test_metrics = evaluate(model, test_loader)
print(f'Test pixel accuracy: {test_metrics["pixel_acc"]*100:.2f}%')
print(f'Test mIoU:           {test_metrics["mIoU"]:.3f}')
for c, name in enumerate(CLASS_NAMES):
    print(f'  {name:18s} IoU: {test_metrics["IoU_per_class"][c]:.3f}')

In [ ]:
# ─── 視覺化預測 ───
model.eval()
with torch.no_grad():
    rgb, mask = next(iter(test_loader))
    rgb_d = rgb.to(device)
    pred = model(rgb_d).argmax(dim=1).cpu()

n = min(4, rgb.size(0))
fig, axes = plt.subplots(3, n, figsize=(4*n, 10))
for i in range(n):
    axes[0, i].imshow(unnorm(rgb[i]));            axes[0, i].axis('off'); axes[0, i].set_title('RGB')
    axes[1, i].imshow(mask_to_color(mask[i].numpy())); axes[1, i].axis('off'); axes[1, i].set_title('GT')
    axes[2, i].imshow(mask_to_color(pred[i].numpy())); axes[2, i].axis('off'); axes[2, i].set_title('Pred')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Instance-level 後處理：connected components → per-instance pass/fail ───
# semantic seg + connected components → 每個獨立 blob 一個 instance
# 若 blob 內含類別 2 像素 → defective
from scipy import ndimage

def instance_metrics(pred, gt, min_area=50):
    """
    pred, gt: H×W numpy, values 0/1/2
    回傳每張圖的: n_gt_instances, n_pred_instances, defect_TP, defect_FP, defect_FN
    """
    # 把零件 region (class 1 or 2) 二值化做 CC
    gt_parts   = (gt   > 0).astype(np.uint8)
    pred_parts = (pred > 0).astype(np.uint8)
    gt_lbl,   n_gt   = ndimage.label(gt_parts)
    pred_lbl, n_pred = ndimage.label(pred_parts)
    
    # 每個 GT instance 是否 defective？
    gt_defects   = {i: (gt[gt_lbl == i] == 2).any() for i in range(1, n_gt+1)}
    pred_defects = {i: (pred[pred_lbl == i] == 2).any() for i in range(1, n_pred+1)}

    # IoU matching：pred instance → 最 overlap 的 gt instance
    TP = FP = FN = 0
    matched_gt = set()
    for pi in range(1, n_pred+1):
        if (pred_lbl == pi).sum() < min_area: continue
        # find best gt match
        overlaps = {}
        pred_mask = (pred_lbl == pi)
        for gi in range(1, n_gt+1):
            iou = (pred_mask & (gt_lbl == gi)).sum() / (pred_mask | (gt_lbl == gi)).sum()
            overlaps[gi] = iou
        if not overlaps:
            FP += 1; continue
        best_gi = max(overlaps, key=overlaps.get)
        if overlaps[best_gi] < 0.3:
            FP += 1; continue
        # match — 比對 defect 預測
        matched_gt.add(best_gi)
        if pred_defects[pi] == gt_defects[best_gi]:
            TP += 1
        else:
            FP += 1
    FN = n_gt - len(matched_gt)
    return n_gt, n_pred, TP, FP, FN

# 跑完整個 test set 算 instance metrics
total_TP = total_FP = total_FN = 0; total_gt = total_pred = 0
model.eval()
with torch.no_grad():
    for rgb, mask in test_loader:
        rgb = rgb.to(device)
        pred = model(rgb).argmax(dim=1).cpu().numpy()
        for i in range(pred.shape[0]):
            n_gt, n_pred, TP, FP, FN = instance_metrics(pred[i], mask[i].numpy())
            total_gt += n_gt; total_pred += n_pred
            total_TP += TP; total_FP += FP; total_FN += FN

precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
recall    = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f'Instance-level pass/fail (含類別正確性):')
print(f'  Total GT instances: {total_gt}, Predicted: {total_pred}')
print(f'  Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}')

In [ ]:
# ─── 存 best model ───
save_path = r'D:\Harrison\中山\大四下\深度學習期末報告\output\stage2_best_model.pt'
torch.save({
    'state_dict': model.state_dict(),
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'best_val_miou': best_miou,
}, save_path)
print(f'Saved → {save_path}')